# 网络编程、进程与线程

本篇对应《Python进阶》课程第 4 天，包含三部分：网络编程、进程和线程。

学习主线：先认识网络通信需要的 IP 地址、端口和协议，再用 `socket` 完成 TCP 通信；随后把“同时处理多个任务”的问题分别交给进程和线程解决。

课程代码均保留在 Markdown 代码围栏中，示例不在本笔记中执行。网络通信示例默认使用本机地址和教学端口，实际运行时需要先启动服务端，再启动客户端。

## 一、网络编程基础

### 1. 网络与网络编程

网络是把多台具有独立功能的计算机，通过通信线路、通信设备、网络管理软件和通信协议连接起来，实现资源共享和信息传递的平台。

用编程语言编写能够在网络中发送和接收数据的程序，称为网络编程。两台计算机要正常通信，必须约定“找到哪台设备、找到哪个程序、按什么规则交流”。这对应网络编程的三个要素：

| 要素 | 作用 | 类比 |
| --- | --- | --- |
| IP 地址 | 定位网络中的设备 | 找到小明的住处 |
| 端口 | 定位设备上运行的进程 | 找到小明所在的房间 |
| 协议 | 规定通信格式和规则 | 双方使用都能听懂的语言 |

只知道 IP 地址只能找到一台设备，还需要端口才能把数据交给设备上的目标程序。

### 2. IP 地址

IP 地址是网络中设备的地址，用来标识并定位一台设备。课程介绍两种版本：

- `IPv4` 是目前最常见的形式，例如 `192.168.1.10`；
- `IPv6` 地址空间更大，是后续网络环境中的重要版本。

IP 地址的作用是找到目标设备，建立后续数据通信。常见检查命令如下：

```bash
# Linux 和 macOS 查看网卡信息
ifconfig

# Windows 查看网卡信息
ipconfig

# 检查是否能访问公网
ping www.baidu.com

# 检查本机网卡和 TCP/IP 协议栈
ping 127.0.0.1
```

`127.0.0.1` 表示本机回环地址。局域网通信时，还要确认两台设备处于可互通的网络中。

### 3. 端口和端口号

一台计算机可以同时运行多个程序。端口是数据传输的通道，端口号是定位端口的数字标识。网络数据通常通过“IP 地址 + 端口号”找到目标进程。

端口号由已经运行的程序使用。编写服务端时，需要选择一个未被占用的端口；如果端口已经被其他程序绑定，调用 `bind()` 会失败。教学示例常用高位端口，例如 `8080`、`9090`，实际项目应根据环境配置。

> IP 负责找到设备，端口负责找到进程，协议负责约定数据怎样传输。

### 4. TCP 协议与连接过程

TCP 是面向连接的、可靠的传输协议。它在发送数据前建立连接，传输过程中保证数据按序到达，并在通信结束后释放连接。

TCP 连接建立通常称为“三次握手”：

1. 客户端发送连接请求；
2. 服务端确认请求并发送自己的连接请求；
3. 客户端再次确认，连接建立。

TCP 连接释放通常称为“四次挥手”：双方分别发送结束确认，保证双方都能完成剩余数据的发送和接收。三次握手解决“能否建立连接”，四次挥手解决“双方如何有序结束连接”。

网络编程中，TCP 只保证字节流可靠到达，不会自动替应用程序划分消息边界。因此应用层还需要约定编码、长度或结束标志。

## 二、Socket 与 TCP 开发流程

### 1. Socket 是什么

`socket`（套接字）是应用程序使用网络协议进行通信的编程接口。Python 的 `socket` 模块把 IP、端口和 TCP/UDP 等底层能力封装成可调用的对象。

TCP 服务端和客户端的职责不同：服务端先创建监听端口，客户端主动发起连接；连接建立后，双方都可以通过套接字发送和接收数据。

### 2. TCP 服务端程序流程

服务端的典型步骤是：

1. 创建 TCP 套接字；
2. 绑定本机 IP 和端口；
3. 调用 `listen()` 进入监听状态；
4. 调用 `accept()` 等待客户端连接；
5. 使用返回的连接套接字收发数据；
6. 关闭连接套接字和监听套接字。

```python
import socket

server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server_socket.bind(("0.0.0.0", 8080))
server_socket.listen(5)

print("服务器正在等待客户端连接")
connection, client_address = server_socket.accept()
print("客户端地址：", client_address)

received = connection.recv(1024)
print("收到：", received.decode("utf-8"))
connection.send("服务端已收到".encode("utf-8"))

connection.close()
server_socket.close()
```

`server_socket` 负责监听，`accept()` 返回的 `connection` 负责与某一个客户端通信。一个监听套接字可以依次或并发接收多个客户端，但每个客户端都要使用自己的连接套接字。

### 3. TCP 客户端程序流程

客户端步骤更短：创建套接字、连接服务端、收发数据、关闭套接字。

```python
import socket

client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_socket.connect(("127.0.0.1", 8080))

client_socket.send("你好，服务器".encode("utf-8"))
received = client_socket.recv(1024)
print("收到：", received.decode("utf-8"))

client_socket.close()
```

运行顺序：先启动服务端，使端口进入监听状态；再启动客户端。`send()` 和 `recv()` 传输的是字节，所以字符串要用 `encode()` 转换，收到后用 `decode()` 转回字符串。

易错点：`recv(1024)` 的参数是本次最多接收的字节数，不代表一次调用就能得到完整业务消息；真实协议通常需要循环接收或约定消息长度。

## 三、多任务、并发与并行

### 1. 为什么需要多任务

单任务程序一次只能处理一件事。例如程序先下载文件，再处理数据，再响应用户，前一步等待时后面的任务无法推进。多任务允许程序在同一段时间内推进多个独立任务，提高等待时间的利用率。

- **并发**：一个时间段内交替推进多个任务，任务可能在不同时间片运行；
- **并行**：多个任务在同一时刻真正同时运行，通常需要多个 CPU 核心。

网络等待、磁盘读写等 I/O 场景适合并发；计算量很大的任务是否能并行，要结合进程、CPU 核心和 Python 运行时限制判断。

## 四、进程

### 1. 进程的概念和作用

进程是正在运行的程序，是操作系统进行资源分配和调度的基本单位。一个程序启动后，操作系统会为它分配独立的内存空间和运行资源。

使用多个进程，可以把多个相对独立的任务交给操作系统调度。进程之间相互隔离，某个进程崩溃时通常不会直接破坏另一个进程的局部变量。

### 2. 创建进程完成多任务

Python 使用 `multiprocessing` 模块创建进程。`Process` 的 `target` 指定子进程执行的函数，`args` 传递位置参数。

```python
import time
from multiprocessing import Process


def download():
    for index in range(3):
        print("下载中", index)
        time.sleep(1)


def calculate():
    for index in range(3):
        print("计算中", index)
        time.sleep(1)


if __name__ == "__main__":
    download_process = Process(target=download)
    calculate_process = Process(target=calculate)

    download_process.start()
    calculate_process.start()

    download_process.join()
    calculate_process.join()
    print("所有子进程执行结束")
```

Windows 创建子进程时必须使用 `if __name__ == "__main__":` 保护入口，避免子进程重复导入主模块并不断创建新的子进程。

### 3. 进程传参

创建 `Process` 时可以使用 `args` 传递元组参数，也可以使用 `kwargs` 传递字典参数：

```python
from multiprocessing import Process


def show_message(name, count):
    for index in range(count):
        print(name, index)


if __name__ == "__main__":
    process = Process(
        target=show_message,
        args=("子进程", 3),
    )
    process.start()
    process.join()
```

`args` 中只有一个参数时仍要写成单元素元组，例如 `args=(value,)`。`start()` 才会真正创建并调度子进程，直接调用 `target()` 只是普通函数调用。

### 4. 获取进程编号

操作系统会为每个进程分配唯一的进程编号 PID。`os.getpid()` 可以获取当前进程编号，`os.getppid()` 可以获取当前进程的父进程编号。

```python
import os
from multiprocessing import Process


def show_pid():
    print("当前进程 PID：", os.getpid())
    print("父进程 PID：", os.getppid())


if __name__ == "__main__":
    print("主进程 PID：", os.getpid())
    process = Process(target=show_pid)
    process.start()
    process.join()
```

父进程创建子进程后，子进程拥有自己的 PID。PID 适合用于日志、调试和定位系统中的具体进程。

### 5. 进程的注意点

#### 进程之间默认不共享全局变量

每个进程拥有独立的内存空间。主进程和子进程即使使用同名全局变量，也不是同一个对象；子进程修改自己的副本，不会自动改变主进程中的值。需要跨进程共享数据时，应使用队列、管道、共享内存或管理器等进程间通信工具，本课先掌握进程隔离这一基本边界。

#### 主进程等待子进程结束

`join()` 会阻塞当前进程，直到对应子进程结束。需要确保所有子任务完成后再执行后续逻辑时，应在子进程启动后调用 `join()`。

```python
if __name__ == "__main__":
    processes = [Process(target=download) for _ in range(2)]
    for process in processes:
        process.start()
    for process in processes:
        process.join()
    print("主进程继续执行")
```

不要把 `join()` 放在第一个进程 `start()` 后、其他进程 `start()` 前，否则可能把原本可以并发的任务变成近似串行。

## 五、线程

### 1. 线程的概念

线程是进程内部的执行单元，也是 CPU 调度的基本单位。一个进程可以包含多个线程，这些线程共享进程的内存和资源，因此创建和切换线程通常比创建进程更轻量。

线程适合处理同一进程内的多个 I/O 任务，例如同时等待多个网络请求。由于线程共享数据，多个线程同时修改同一对象时必须考虑同步问题。

### 2. 创建线程完成多任务

Python 使用 `threading.Thread` 创建线程。调用 `start()` 后线程开始执行目标函数，`join()` 用于等待线程结束。

```python
import threading
import time


def download():
    for index in range(3):
        print("下载中", index)
        time.sleep(1)


def calculate():
    for index in range(3):
        print("计算中", index)
        time.sleep(1)


if __name__ == "__main__":
    download_thread = threading.Thread(target=download)
    calculate_thread = threading.Thread(target=calculate)

    download_thread.start()
    calculate_thread.start()

    download_thread.join()
    calculate_thread.join()
    print("所有线程执行结束")
```

线程使用 `args` 和 `kwargs` 传递目标函数参数，形式与 `Process` 类似。线程对象启动后，不能重复调用 `start()`；需要再次执行时应创建新的线程对象。

### 3. 线程共享数据与同步

同一进程中的线程共享全局变量和堆对象，这让线程之间传递数据很方便，也带来竞态条件：一个线程读取数据后，另一个线程可能在中间修改它，最终结果取决于调度顺序。

可以使用 `threading.Lock` 保护临界区。获得锁后执行共享数据的读写，完成后释放锁；使用 `with` 可以自动释放锁：

```python
import threading

counter = 0
counter_lock = threading.Lock()


def increase():
    global counter
    for _ in range(1000):
        with counter_lock:
            counter += 1


if __name__ == "__main__":
    threads = [threading.Thread(target=increase) for _ in range(2)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    print(counter)
```

锁应只包围必要的共享操作，否则会降低并发度；多个锁之间还要保持一致的加锁顺序，避免死锁。

## 六、进程与线程的比较

| 对比项 | 进程 | 线程 |
| --- | --- | --- |
| 资源 | 拥有相对独立的内存空间 | 共享所属进程的内存和资源 |
| 创建和切换 | 成本较高 | 成本较低 |
| 数据共享 | 默认不共享，需要进程间通信 | 共享方便，但需要同步保护 |
| 隔离性 | 较强，一个进程故障通常影响较小 | 较弱，一个线程异常可能影响整个进程 |
| 适用场景 | 需要隔离、CPU 密集或独立任务 | 同一进程内的 I/O 并发、共享数据任务 |

选择时先看任务边界：需要独立资源和故障隔离时考虑进程；需要轻量并发并且任务共享内存时考虑线程。使用线程访问共享数据时，要同步保护；使用进程时，要设计进程间通信方式。

## 七、易错点与本日回顾

- 网络通信至少要明确 IP、端口和协议；
- `socket` 收发的是字节，字符串要进行 `encode()` 和 `decode()`；
- TCP 是字节流协议，一次 `recv()` 不一定对应一次 `send()`；
- 服务端先 `bind()`、`listen()`，再 `accept()`；客户端使用 `connect()`；
- `Process.start()` 和 `Thread.start()` 才会启动并发任务，直接调用目标函数不会创建新执行单元；
- Windows 多进程代码要放在 `if __name__ == "__main__":` 中；
- `join()` 用于等待任务结束，放置位置会影响并发效果；
- 进程默认隔离内存，线程共享进程内存；
- 线程共享数据时要用锁保护临界区，避免竞态条件；
- 无论进程还是线程，资源都要在任务结束后正确关闭，例如 socket、文件和锁。